In [ ]:
from kooplearn.metrics import spectral_bias, rrr_truncation, kdmd_truncation
from sklearn.metrics.pairwise import pairwise_kernels
from kooplearn.kernel import KernelRidge
from collections import defaultdict
from tqdm import tqdm
import numpy as np
from scipy.integrate import romb
from kooplearn.datasets import make_prinz_potential

n_train_samples = 5001
n_components = 10
gamma = 1.0
sigma = 2.0


def prinz_potential(x):
    return 4 * (
        x**8
        + 0.8 * np.exp(-80 * (x**2))
        + 0.2 * np.exp(-80 * ((x - 0.5) ** 2))
        + 0.5 * np.exp(-40 * ((x + 0.5) ** 2))
    )


def compute_boltzmann_density(x, gamma, sigma):
    beta = 2 * gamma / (sigma**2)
    pdf = np.exp(-beta * prinz_potential(x))
    total_mass = romb(pdf, dx=x[1] - x[0])
    return pdf / total_mass


x = np.linspace(-2, 2, 2048 + 1)
density = compute_boltzmann_density(x, gamma, sigma)
data = make_prinz_potential(X0=0, n_steps=int(7e5), gamma=gamma, sigma=sigma)


def fit_and_estimate(reduced_rank, x, density, random_state):
    subsample = 100  # long trajectories for sampling Boltzmann distribution

    # Parameters of Langevin equation
    gamma = 1.0  # damping coefficient
    sigma = 2.0  # noise coefficient, proportional to temperature
    data = make_prinz_potential(
        X0=0, n_steps=int(7e5), gamma=gamma, sigma=sigma, random_state=random_state
    )

    # Sample new trajectories
    data = data.iloc[
        ::subsample  # don't need all data
    ]

    data = data[:n_train_samples]

    # Model definition
    model = KernelRidge(
        n_components=5,
        reduced_rank=reduced_rank,
        gamma=12.5,
        kernel="rbf",
        alpha=1e-6,
        random_state=random_state,
    )

    # Fit and estimate eigenfunctions
    model.fit(data)  # fit transfer op model
    values, functions = model.eig(
        eval_right_on=x
    )  # (right) eigenvalue estimation, evaluate on array x
    sort_perm = np.flip(np.argsort(np.abs(values)))  # Order decreasingly
    values, functions = values[sort_perm], functions[:, sort_perm]
    functions = normalize_eigenfunctions(functions, x, density)
    return functions


def normalize_eigenfunctions(functions, x, density):
    abs2_eigfun = (np.abs(functions) ** 2).T  # f(x)**2
    abs2_eigfun *= density  # Compute the norm with respect to the Boltzmann measure.
    dx = x[1] - x[0]
    funcs_norm = np.sqrt(romb(abs2_eigfun, dx=dx, axis=1))  # Norms
    functions *= funcs_norm**-1.0  # Normalize
    return functions


X_train = data[["x"]].to_numpy()
K = pairwise_kernels(X_train, X_train, metric="rbf", gamma=12.5)

r = n_components
trunc_kdmd = kdmd_truncation(K, r)
trunc_rrr = rrr_truncation(K, r)

biases = {"Principal Components (kDMD)": [], "Reduced Rank": []}

for method, reduced_rank in zip(
    ["Principal Components (kDMD)", "Reduced Rank"], [False, True]
):
    for i in tqdm(range(10)):
        values, functions = fit_and_estimate(reduced_rank, x, density, i)

        if method == "Principal Components (kDMD)":
            bias = spectral_bias(
                functions=functions,
                covariance=K,
                truncation=trunc_kdmd,
            )
            biases[method].append(bias)

        else:
            bias = spectral_bias(
                functions=functions,
                covariance=K,
                truncation=trunc_rrr,
            )
            biases[method].append(bias)

    print(
        f"{method}: mean bias = {np.mean(biases[method]):.6f}, "
        f"std = {np.std(biases[method]):.6f}"
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def analyse_bias(records, out_prefix="spectral_bias"):
    df = pd.DataFrame(records).copy()

    if "eigenvalue_error" not in df.columns and {"true_eig", "est_eig"}.issubset(
        df.columns
    ):
        df["eigenvalue_error"] = np.abs(df["est_eig"] - df["true_eig"])

    summary = df.groupby(["kernel", "method", "eigenfunction_id"], as_index=False).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        err_mean=("eigenvalue_error", "mean"),
        err_std=("eigenvalue_error", "std"),
        dist_mean=("metric_distortion", "mean")
        if "metric_distortion" in df.columns
        else ("spectral_bias", "mean"),
        trunc_mean=("truncation", "mean")
        if "truncation" in df.columns
        else ("spectral_bias", "mean"),
    )

    corr_rows = []
    for (kernel, method, eig_id), g in df.groupby(
        ["kernel", "method", "eigenfunction_id"]
    ):
        if len(g) >= 2:
            pear = g["spectral_bias"].corr(g["eigenvalue_error"], method="pearson")
            spear = g["spectral_bias"].corr(g["eigenvalue_error"], method="spearman")
        else:
            pear = np.nan
            spear = np.nan
        corr_rows.append(
            {
                "kernel": kernel,
                "method": method,
                "eigenfunction_id": eig_id,
                "pearson": pear,
                "spearman": spear,
                "n": len(g),
            }
        )
    corr_df = pd.DataFrame(corr_rows)

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_correlations.csv", index=False)
    df.to_csv(f"{out_prefix}_records.csv", index=False)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, method), g in df.groupby(["kernel", "method"]):
        ax.scatter(
            g["spectral_bias"],
            g["eigenvalue_error"],
            s=20,
            alpha=0.7,
            label=f"{kernel} / {method}",
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Eigenvalue error")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs eigenvalue error")
    fig.tight_layout()
    fig.savefig(f"{out_prefix}_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    return df, summary, corr_df
